# Phase 8 — Huấn luyện Model C (Dev-only)
Notebook không nhận ViLexNorm Test hoặc `outputs/evaluation/` làm input.

In [ ]:
from pathlib import Path
import shutil, subprocess

REPO = Path('/kaggle/working/VisolexNorm')
SOURCE_REF = 'main'
URL = 'https://github.com/AIVIETNAM-AIO-DinhBao/VisolexNorm.git'
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(['git', 'clone', '--depth', '1', '--branch', SOURCE_REF, URL, str(REPO)], check=True)
source_commit = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', source_commit], check=True)
print(f'?? kh?a source revision refactor: {source_commit}')

In [ ]:
%cd {REPO}
!pip install -q -r requirements-kaggle.txt
import torch
assert torch.cuda.is_available(), 'Hãy bật Accelerator GPU trong Notebook options'
print(torch.cuda.get_device_name(0))

## Input Dataset
Dataset private phải giữ cấu trúc `checkpoints/model_a`, `data/processed` và `outputs/expanded_review`.

In [ ]:
MOUNT = Path('/kaggle/input/visolexnorm-phase8-training')
WORK = Path('/kaggle/working')
ARCHIVE = MOUNT/'phase8_training_input.zip'
DATA = WORK/'phase8_training_input' if ARCHIVE.is_file() else MOUNT
if ARCHIVE.is_file():
    if DATA.exists():
        shutil.rmtree(DATA)
    shutil.unpack_archive(ARCHIVE, DATA)
required = [
    DATA/'checkpoints/model_a/config.json',
    DATA/'data/processed/vilexnorm_train.jsonl',
    DATA/'data/processed/vilexnorm_dev.jsonl',
    DATA/'data/processed/visolex_weak_labeled_expanded.jsonl',
    DATA/'outputs/expanded_review/artifact_manifest.json',
]
missing = [str(path) for path in required if not path.is_file()]
assert not missing, f'Thiếu input: {missing}'
assert not (DATA/'data/processed/vilexnorm_test.jsonl').exists(), 'Không upload Test vào dataset Phase 8'
assert not (DATA/'outputs/evaluation').exists(), 'Không upload outputs/evaluation vào dataset Phase 8'

In [ ]:
!python -m pytest tests/training/test_model_c_reports.py tests/training/test_validation.py -q
!python -m scripts.training build-mixture --model model_c --repo-root {DATA} --config {REPO}/configs/model_c_config.json --output {WORK}/training_mixture_manifest.json

## Smoke gate — 200 gold + 200 pseudo

In [ ]:
!python -m scripts.training train --model model_c --model-a-checkpoint {DATA}/checkpoints/model_a --data-dir {DATA}/data/processed --mixture-manifest {WORK}/training_mixture_manifest.json --config {REPO}/configs/model_c_config.json --work-dir {WORK}/smoke --smoke-test
import json
smoke_path = WORK/'smoke/outputs/model_c/smoke_test.json'
smoke = json.loads(smoke_path.read_text())
assert smoke['passed'] and smoke['composition'] == {'gold': 200, 'pseudo': 200}, smoke
assert smoke['checkpoint_reload'] and smoke['checkpoint_inventory_verified'], smoke
assert smoke['test_inputs_loaded'] is False, smoke
smoke

## Full run — 8 epoch theo mixture đã freeze

In [ ]:
!python -m scripts.training train --model model_c --model-a-checkpoint {DATA}/checkpoints/model_a --data-dir {DATA}/data/processed --mixture-manifest {WORK}/training_mixture_manifest.json --config {REPO}/configs/model_c_config.json --work-dir {WORK} --smoke-report {smoke_path}

In [ ]:
import json, shutil
metrics = json.loads((WORK/'outputs/model_c/dev_metrics.json').read_text())
config = json.loads((WORK/'outputs/model_c/train_config.json').read_text())
assert metrics['dev_examples'] == 1050
assert config['test_inputs_loaded'] is False
shutil.make_archive('/kaggle/working/model_c_artifacts', 'zip', WORK, 'outputs/model_c')
shutil.make_archive('/kaggle/working/model_c_checkpoint', 'zip', WORK, 'checkpoints/model_c')
print('Tải model_c_artifacts.zip và model_c_checkpoint.zip trong Output.')